# jlens-audit — persistent kernel

**Rule**: the SETUP cell runs ONCE. Never restart the kernel without human validation. Every experiment writes to `results/` and `figs/`.

In [ ]:
# SETUP — once only
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.load_model import load, layers, get_resid
tok, model = load()
print('model loaded —', model.config.num_hidden_layers, 'layers ; scanned:', layers())

In [ ]:
# LENSES — after resolving the # ADAPTER points in src/lens.py by reading lenses/README.md
from src.lens import load_all
lenses = load_all()
print({k: len(v.maps) for k, v in lenses.items()})

## Step 2 — lens go/no-go, then soft conformity
**Go/no-go (binary, decisive)**: `identity_check` (the anchor `J_62 = I` must reproduce the logit lens) + `orientation_check` (overlap at the sub-anchor: catches a transpose, which the anchor cannot see). If either breaks → STOP, the lens setup is wrong.

**Soft conformity** ("sushi → Japan"), order of magnitude: R-lens from the early layers, J-lens later, logit lens late or never.

In [ ]:
from src.validate import identity_check, orientation_check, smoke
identity_check(lenses)      # STOP on failure
orientation_check(lenses)   # STOP on failure
smoke(lenses)

## Step 3 — quantitative validation (multihop pass@k)

In [ ]:
from src.validate import pass_at_k
pass_at_k(lenses)   # -> results/validation_multihop.json, figs/validation_multihop.png

## Step 4 — model capability on the pilot pairs (test 2)

In [ ]:
from src import capability
capability.main('pairs_pilot.jsonl')   # target >= 80% in-the-clear detection per family

## Step 5 — timing test on one pair (test 3)

In [ ]:
import json, time
from src.scan import scan_text
p = json.loads(open('../data/pairs_pilot.jsonl').readline())
t0 = time.time(); out, toks = scan_text(p['anomalous'], lenses); print(f'{time.time()-t0:.1f}s for {len(toks)} positions')
# inspect a few positions around the anomaly by hand:
from src.serialize import serialize
print('\n'.join(serialize(out['jlens']).split('\n')[:15]))

## Log
Before closing: an entry in `experiments.md` (done / verified / doubt / next step).